# Movie Review Sentiment Analysis
**Stephanie Nord**

## Project Overview

This project compares TextBlob and VADER for classifying movie reviews as positive or negative. Predictions are evaluated against labeled reviews using accuracy and confusion matrices.

The notebook also explores text preprocessing and creates bag-of-words and TF-IDF feature matrices as preparation for future supervised modeling. No custom classifier is trained in this analysis.

## Setup and Data Loading

Import the required libraries and load the labeled movie reviews from `labeledTrainData.tsv`. Keep this file in the same folder as the notebook.

The commented setup commands below only need to run if the required packages or language resources are missing.

In [1]:
# %pip install textblob

In [2]:
# !python -m textblob.download_corpora

In [3]:
import pandas as pd
import re
from textblob import TextBlob

from sklearn.metrics import accuracy_score, confusion_matrix

# Download required NLTK data (run once)
import nltk
# nltk.download('vader_lexicon')

# Load TSV dataset
df = pd.read_csv('labeledTrainData.tsv', sep='\t')

# Preview data
df.head()

,id,sentiment,review
0,5814_8,1,With all this stuff going down at the moment w...
1,2381_9,1,"\The Classic War of the Worlds\"" by Timothy Hi..."
2,7759_3,0,The film starts with a manager (Nicholas Bell)...
3,3630_4,0,It must be assumed that those who praised this...
4,9495_8,1,Superbly trashy and wondrously unpretentious 8...


The dataset was successfully loaded from a TSV file and contains movie reviews along with their sentiment labels. The data includes a review column with text and a sentiment column indicating whether the review is positive or negative.

## Dataset Overview

The dataset contains movie review text in `review` and sentiment labels in `sentiment`, where 1 represents positive and 0 represents negative.

This dataset contains movie reviews labeled as positive or negative. The data was loaded from a TSV file and includes a text column (`review`) and a sentiment label (`sentiment`), where 1 indicates positive and 0 indicates negative.

The output above should show a dataframe with the review text and its true sentiment label.

In [4]:
print('Data shape:', df.shape)

print('\nSentiment counts:')
print(df['sentiment'].value_counts())

Data shape: (25000, 3)

Sentiment counts:
sentiment
1    12500
0    12500
Name: count, dtype: int64


The dataset contains 12,500 positive and 12,500 negative reviews, making it balanced. This is important because it provides a fair baseline for evaluating model performance, where random guessing would result in approximately 50% accuracy.

In [5]:
df['label'] = df['sentiment'].map({1: 'pos', 0: 'neg'})

### Answer: How many positive and negative reviews are there?
The dataset contains 12,500 positive and 12,500 negative reviews, making it balanced. This provides a fair baseline for evaluating model performance, as random guessing would yield approximately 50% accuracy.

## TextBlob Sentiment Classification
A polarity score greater than or equal to 0 is classified as positive, and a score less than 0 is classified as negative.


In [6]:
def textblob_label(text):
    polarity = TextBlob(text).sentiment.polarity
    return 'pos' if polarity >= 0 else 'neg'

df['textblob_pred'] = df['review'].apply(textblob_label)

### TextBlob Evaluation

Compare the predicted sentiment labels with the dataset’s reference labels.

In [7]:
accuracy = accuracy_score(df['label'], df['textblob_pred'])
print("TextBlob accuracy:", accuracy)

TextBlob accuracy: 0.68524


### TextBlob Results and Limitations

TextBlob achieved 68.52% accuracy, exceeding the 50% baseline for this balanced dataset by 18.52 percentage points. This evaluates TextBlob’s existing sentiment analyzer; no model was trained on these reviews.

The classification rule assigns zero-polarity reviews to the positive class. Reviewing misclassified examples would help explain the errors, since accuracy alone cannot establish whether sarcasm or other language patterns caused them.

### Interpretation

TextBlob uses a pretrained polarity scoring system rather than a model trained on this dataset. The results show that it performs better than random guessing, indicating that it is able to capture meaningful sentiment patterns, although its performance is moderate.

## VADER Sentiment Classification

VADER provides a second sentiment analysis approach for comparison. This analysis classifies compound scores greater than or equal to zero as positive and scores below zero as negative. As with the TextBlob rule, zero scores are assigned to the positive class.

In [9]:
# %pip install nltk

In [10]:
import nltk
# nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\sfnor\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [11]:
# Import VADER
from nltk.sentiment import SentimentIntensityAnalyzer

# Initialize analyzer
sia = SentimentIntensityAnalyzer()

# Define labeling function
def vader_label(text):
    compound = sia.polarity_scores(text)['compound']
    return 'pos' if compound >= 0 else 'neg'

# Apply to dataset
df['vader_pred'] = df['review'].apply(vader_label)

# Accuracy
vader_accuracy = accuracy_score(df['label'], df['vader_pred'])
print("VADER accuracy:", round(vader_accuracy, 4))

# Confusion matrix
cm_vader = confusion_matrix(df['label'], df['vader_pred'], labels=['pos', 'neg'])
print("\nConfusion matrix (rows = true, columns = predicted):")
print(pd.DataFrame(cm_vader, index=['true_pos', 'true_neg'], columns=['pred_pos', 'pred_neg']))

# Compare to baseline
print("\nVADER better than random guessing?:", vader_accuracy > 0.5)

VADER accuracy: 0.6936

Confusion matrix (rows = true, columns = predicted):
          pred_pos  pred_neg
true_pos     10657      1843
true_neg      5818      6682

VADER better than random guessing?: True


### VADER Results and Limitations

VADER achieved 69.36% accuracy. It correctly classified 10,657 of 12,500 positive reviews (85.3%) and 6,682 of 12,500 negative reviews (53.5%).

The confusion matrix shows substantially stronger performance on positive reviews. VADER incorrectly classified 5,818 negative reviews as positive, highlighting an imbalance that overall accuracy alone does not reveal.

## Part 2: Prepping Text for a Custom Model

The following steps clean and transform the review text so it can be used later in a machine learning model.


In [12]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Download stopwords (run once)
# nltk.download('stopwords')

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [stemmer.stem(word) for word in tokens]
    return ' '.join(tokens)

df['review_stemmed'] = df['review'].apply(preprocess_text)
df[['review', 'review_stemmed']].head()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sfnor\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,review,review_stemmed
0,With all this stuff going down at the moment w...,stuff go moment mj ive start listen music watc...
1,"\The Classic War of the Worlds\"" by Timothy Hi...",classic war world timothi hine entertain film ...
2,The film starts with a manager (Nicholas Bell)...,film start manag nichola bell give welcom inve...
3,It must be assumed that those who praised this...,must assum prais film greatest film opera ever...
4,Superbly trashy and wondrously unpretentious 8...,superbl trashi wondrous unpretenti exploit hoo...


### Preprocessing Results and Limitations

The processed reviews are lowercase, with punctuation and numbers removed. Stopwords are filtered out, and Porter stemming reduces words to shortened stems that are not always complete words.

These transformations prepare the text for bag-of-words and TF-IDF features, but they can also remove useful sentiment information. For example, the default stopword list removes negations such as “not.” This preprocessing is a feature preparation exercise; TextBlob and VADER were evaluated separately using the original reviews.

## Bag-of-Words Features
Each row represents one movie review, and each column represents a unique word in the processed text.


In [13]:
from sklearn.feature_extraction.text import CountVectorizer

count_vectorizer = CountVectorizer()
bow_matrix = count_vectorizer.fit_transform(df['review_stemmed'])

print("Bag-of-words matrix shape:", bow_matrix.shape)
print("Number of rows in dataframe:", df.shape[0])
print("Rows match original dataframe?:", bow_matrix.shape[0] == df.shape[0])

Bag-of-words matrix shape: (25000, 89468)
Number of rows in dataframe: 25000
Rows match original dataframe?: True


## TF-IDF Features

TF-IDF weights terms according to their frequency within a review and their prevalence across the dataset. Terms appearing in many reviews receive lower inverse-document-frequency weights.

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(df['review_stemmed'])

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Matches bag-of-words dimensions?:", tfidf_matrix.shape == bow_matrix.shape)

TF-IDF matrix shape: (25000, 89468)
Matches bag-of-words dimensions?: True


### Feature Matrix Interpretation

Both matrices contain 25,000 rows and 89,468 features. Each row represents a review, and each column represents a token retained by the vectorizer after preprocessing. Bag-of-words stores term counts, while TF-IDF stores weighted values.

These matrices demonstrate feature preparation; no classifier is trained with them here. For future supervised modeling, the reviews should be split into training and test sets before fitting the vectorizers. The fitted vectorizers would then transform the test reviews without learning from them.

## Conclusion

This project evaluated TextBlob and VADER for binary movie review sentiment classification. TextBlob achieved approximately 68.5% accuracy, while VADER achieved 69.36%. Both exceeded the 50% baseline for the balanced dataset. However, VADER recognized positive reviews much more successfully than negative reviews, demonstrating why overall accuracy should be considered alongside class-level results.

The project also prepared bag-of-words and TF-IDF features for future modeling. Preprocessing simplified the text but introduced limitations, including the removal of negation words that can affect sentiment. Future work could preserve negations, examine misclassified reviews, and compare the existing analyzers with a supervised classifier evaluated on held-out data.